# CosySim Router v3 — Unsloth QLoRA Fine-Tuning

Fine-tunes `Qwen/Qwen2.5-0.5B-Instruct` on the 2080-example router_v3 dataset.

**Model**: 0.5B params → fits T4 (16GB) with 4-bit QLoRA with headroom.

**Output**: LoRA adapter + merged full model → download as zip.

**Time**: ~15-20 min on T4 (~4 Colab compute units)

---

## Instructions
1. Upload `router_v3_train.jsonl` and `router_v3_val.jsonl` when prompted (Cell 3)
2. Run all cells in order
3. Download `router_v3_adapter.zip` from the last cell
4. Extract to `training/models/router_v3_final/` on your local machine

In [ ]:
# Cell 1: Verify GPU and install Unsloth
import subprocess, sys

# Check GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Cell 2: Install Unsloth + dependencies
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
print('Installation complete')

In [ ]:
# Cell 3: Upload dataset files
from google.colab import files
import json, pathlib

print('Upload router_v3_train.jsonl and router_v3_val.jsonl')
uploaded = files.upload()

for fname in uploaded:
    print(f'Uploaded: {fname} ({len(uploaded[fname])} bytes)')

# Verify
train_path = pathlib.Path('router_v3_train.jsonl')
val_path   = pathlib.Path('router_v3_val.jsonl')

assert train_path.exists(), 'Missing router_v3_train.jsonl'
assert val_path.exists(),   'Missing router_v3_val.jsonl'

train_data = [json.loads(l) for l in train_path.read_text().splitlines() if l.strip()]
val_data   = [json.loads(l) for l in val_path.read_text().splitlines() if l.strip()]

print(f'Train: {len(train_data)} examples')
print(f'Val:   {len(val_data)} examples')
print(f'Sample: {train_data[0]}')

In [ ]:
# Cell 4: Load model with Unsloth 4-bit QLoRA
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 512
DTYPE = None   # auto-detect (bfloat16 on A100, float16 on T4)
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'Qwen/Qwen2.5-0.5B-Instruct',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

print(f'Model loaded: {model.config.name_or_path}')
print(f'Parameters:   {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

In [ ]:
# Cell 5: Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                     # LoRA rank
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                       'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} ({100*trainable/total:.2f}% of {total/1e6:.1f}M)')

In [ ]:
# Cell 6: Prepare dataset for SFT
from datasets import Dataset

ALPACA_TEMPLATE = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token

def format_example(example):
    text = ALPACA_TEMPLATE.format(
        instruction = example['instruction'],
        input       = example['input'],
        output      = example['output'],
    ) + EOS_TOKEN
    return {'text': text}

train_dataset = Dataset.from_list(train_data).map(format_example, batched=False)
val_dataset   = Dataset.from_list(val_data).map(format_example, batched=False)

print(f'Train dataset: {len(train_dataset)} examples')
print(f'Val dataset:   {len(val_dataset)} examples')
print(f'Sample text preview:\n{train_dataset[0]["text"][:300]}...')

In [ ]:
# Cell 7: Configure SFT Trainer and train
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    dataset_text_field = 'text',
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4,   # effective batch = 32
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 20,
        evaluation_strategy = 'steps',
        eval_steps = 100,
        save_strategy = 'steps',
        save_steps = 100,
        save_total_limit = 2,
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        lr_scheduler_type = 'cosine',
        seed = 42,
        output_dir = 'router_v3_checkpoints',
        report_to = 'none',
        load_best_model_at_end = True,
        metric_for_best_model = 'eval_loss',
    ),
)

print('Starting training...')
trainer_stats = trainer.train()
print(f'Training complete!')
print(f'  Runtime: {trainer_stats.metrics["train_runtime"]:.0f}s')
print(f'  Samples/sec: {trainer_stats.metrics["train_samples_per_second"]:.1f}')
print(f'  Final loss: {trainer_stats.metrics["train_loss"]:.4f}')

In [ ]:
# Cell 8: Evaluate accuracy on validation set
from unsloth import FastLanguageModel
import json

FastLanguageModel.for_inference(model)

EVAL_TEMPLATE = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
"""

correct = 0
total   = min(100, len(val_data))  # eval first 100 examples

for example in val_data[:total]:
    prompt = EVAL_TEMPLATE.format(
        instruction = example['instruction'],
        input       = example['input'],
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=16, 
                                  temperature=0.1, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
    pred = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], 
                             skip_special_tokens=True).strip().split('\n')[0].strip()
    if pred.lower() == example['output'].lower():
        correct += 1

accuracy = correct / total * 100
print(f'Validation accuracy: {correct}/{total} = {accuracy:.1f}%')

# Save metrics
metrics = {
    'val_accuracy': accuracy,
    'val_correct': correct,
    'val_total': total,
    'train_loss': trainer_stats.metrics.get('train_loss'),
    'model': 'Qwen2.5-0.5B-Instruct',
    'dataset': 'router_v3',
    'train_examples': len(train_data),
}
import json
with open('router_v3_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

In [ ]:
# Cell 9: Save LoRA adapter + merge to 16-bit
import os, zipfile, pathlib

ADAPTER_DIR = 'router_v3_adapter'
MERGED_DIR  = 'router_v3_merged'

# Save LoRA adapter only (small, fast)
print('Saving LoRA adapter...')
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Merge to 16-bit full model (for local use without PEFT)
print('Merging to 16-bit...')
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')

# Copy metrics
import shutil
shutil.copy('router_v3_metrics.json', f'{ADAPTER_DIR}/metrics.json')
shutil.copy('router_v3_metrics.json', f'{MERGED_DIR}/metrics.json')

print(f'Adapter saved to {ADAPTER_DIR}/')
print(f'Merged model saved to {MERGED_DIR}/')

# List files
for d in [ADAPTER_DIR, MERGED_DIR]:
    files_list = list(pathlib.Path(d).rglob('*'))
    total_size = sum(f.stat().st_size for f in files_list if f.is_file()) / 1024**2
    print(f'{d}: {len(files_list)} files, {total_size:.1f} MB')

In [ ]:
# Cell 10: Package and download
import zipfile, pathlib
from google.colab import files

def zip_dir(src_dir, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in pathlib.Path(src_dir).rglob('*'):
            if f.is_file():
                zf.write(f, f.relative_to(src_dir))
    size_mb = pathlib.Path(zip_path).stat().st_size / 1024**2
    print(f'Created {zip_path}: {size_mb:.1f} MB')

# Zip adapter (smaller — for local training/merging later)
zip_dir('router_v3_adapter', 'router_v3_adapter.zip')

# Zip merged model (larger but ready to use)
zip_dir('router_v3_merged', 'router_v3_merged.zip')

# Also download standalone metrics
print('\nDownloading files...')
files.download('router_v3_metrics.json')
files.download('router_v3_adapter.zip')
# Uncomment to download larger merged model (>1GB):
# files.download('router_v3_merged.zip')

print('Done! Extract router_v3_adapter.zip to training/models/router_v3_final/')

## After Download — Local Integration

1. Extract `router_v3_adapter.zip` to `training/models/router_v3_final/`
2. Run the auto-promote script:
   ```bash
   python training/deploy_router.py --adapter training/models/router_v3_final --promote
   ```
3. Verify with:
   ```bash
   python training/smoke_test.py --model router_v3
   ```
4. Check Nexus for the benchmark entry

---

**Expected results**: ~85-92% val accuracy on the 16-class routing task

**Compute cost**: ~4 Colab compute units on T4